# 🔍 Fake Profile Detection in Social Media
**Algorithm:** Random Forest Classifier  
**Dataset:** Synthetic dataset mimicking Kaggle Social Media Profile Dataset  
**Goal:** Classify social media profiles as *Real (0)* or *Fake (1)*

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)
import warnings
warnings.filterwarnings('ignore')

COLORS = {'fake': '#E24B4A', 'real': '#1D9E75', 'neutral': '#534AB7', 'bar': '#3266ad'}
print('Libraries loaded ✓')

## 2. Generate Synthetic Dataset

We generate 5,000 profiles with realistic feature distributions for fake vs real accounts.  
**Features used:**
| Feature | Description |
|---|---|
| `profile_pic` | Has a profile photo (0/1) |
| `nums_in_username` | Count of digits in username |
| `fullname_words` | Words in full name |
| `fullname_nums` | Numbers in full name |
| `bio_length` | Character count in bio |
| `external_url` | Has external URL (0/1) |
| `private` | Account is private (0/1) |
| `posts` | Total posts |
| `followers` | Follower count |
| `follows` | Following count |
| `account_age_days` | Account age in days |
| `follower_follow_ratio` | followers / (follows + 1) |
| `posts_per_day` | posts / (account_age_days + 1) |

In [ ]:
def generate_dataset(n_samples=5000, random_state=42):
    np.random.seed(random_state)
    n_fake = n_samples // 2
    n_real = n_samples - n_fake

    def make_profiles(n, fake=False):
        if fake:
            return {
                'profile_pic':        np.random.choice([0, 1], n, p=[0.6, 0.4]),
                'nums_in_username':   np.random.randint(2, 10, n),
                'fullname_words':     np.random.randint(0, 2, n),
                'fullname_nums':      np.random.randint(0, 5, n),
                'bio_length':         np.random.randint(0, 30, n),
                'external_url':       np.random.choice([0, 1], n, p=[0.7, 0.3]),
                'private':            np.random.choice([0, 1], n, p=[0.8, 0.2]),
                'posts':              np.random.randint(0, 20, n),
                'followers':          np.random.randint(0, 300, n),
                'follows':            np.random.randint(500, 7500, n),
                'account_age_days':   np.random.randint(1, 300, n),
            }
        else:
            return {
                'profile_pic':        np.random.choice([0, 1], n, p=[0.05, 0.95]),
                'nums_in_username':   np.random.randint(0, 3, n),
                'fullname_words':     np.random.randint(1, 4, n),
                'fullname_nums':      np.random.randint(0, 2, n),
                'bio_length':         np.random.randint(20, 160, n),
                'external_url':       np.random.choice([0, 1], n, p=[0.4, 0.6]),
                'private':            np.random.choice([0, 1], n, p=[0.5, 0.5]),
                'posts':              np.random.randint(10, 500, n),
                'followers':          np.random.randint(100, 10000, n),
                'follows':            np.random.randint(50, 2000, n),
                'account_age_days':   np.random.randint(180, 3000, n),
            }

    fake_df = pd.DataFrame(make_profiles(n_fake, fake=True)); fake_df['label'] = 1
    real_df = pd.DataFrame(make_profiles(n_real, fake=False)); real_df['label'] = 0
    df = pd.concat([fake_df, real_df], ignore_index=True).sample(frac=1, random_state=random_state)
    df['follower_follow_ratio'] = df['followers'] / (df['follows'] + 1)
    df['posts_per_day']         = df['posts'] / (df['account_age_days'] + 1)
    return df

df = generate_dataset(5000)
print(f'Dataset shape: {df.shape}')
print(f"Class balance:\n{df['label'].value_counts().rename({0: 'Real', 1: 'Fake'})}")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics by class
df.groupby('label')[['followers', 'follows', 'posts', 'bio_length',
                      'follower_follow_ratio', 'account_age_days']].mean().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor='#F8F8F7')
axes = axes.flatten()

plot_features = ['followers', 'follows', 'bio_length',
                 'follower_follow_ratio', 'posts', 'account_age_days']
clips         = [3000, 7000, 160, 20, 300, 3000]

for ax, feat, clip in zip(axes, plot_features, clips):
    ax.set_facecolor('#F8F8F7')
    for label, color, name in [(1, COLORS['fake'], 'Fake'), (0, COLORS['real'], 'Real')]:
        data = df[df['label'] == label][feat].clip(upper=clip)
        ax.hist(data, bins=35, alpha=0.65, color=color, label=name, edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend()

plt.suptitle('Feature Distributions: Real vs Fake Profiles', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8), facecolor='#F8F8F7')
ax.set_facecolor('#F8F8F7')
corr = df.drop(columns=['label']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.4, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocessing & Train/Test Split

In [ ]:
FEATURES = [
    'profile_pic', 'nums_in_username', 'fullname_words', 'fullname_nums',
    'bio_length', 'external_url', 'private', 'posts', 'followers', 'follows',
    'account_age_days', 'follower_follow_ratio', 'posts_per_day'
]

X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Features: {len(FEATURES)}')

## 5. Train Random Forest Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,      # 200 trees
    max_depth=12,          # max tree depth
    min_samples_split=5,   # min samples to split a node
    min_samples_leaf=2,    # min samples at a leaf
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold scores: {np.round(cv_scores, 4)}')

## 6. Model Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#F8F8F7')

# Confusion Matrix
axes[0].set_facecolor('#F8F8F7')
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', ax=axes[0],
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
            linewidths=0.5, cbar=False, annot_kws={'size': 14})
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
axes[1].set_facecolor('#F8F8F7')
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color=COLORS['neutral'], lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
axes[1].fill_between(fpr, tpr, alpha=0.08, color=COLORS['neutral'])
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# Prediction probability histogram
axes[2].set_facecolor('#F8F8F7')
axes[2].hist(y_proba[y_test==0], bins=30, alpha=0.65, color=COLORS['real'], label='Real', edgecolor='white')
axes[2].hist(y_proba[y_test==1], bins=30, alpha=0.65, color=COLORS['fake'], label='Fake', edgecolor='white')
axes[2].axvline(0.5, color='#2C2C2A', linestyle='--', lw=1.5, label='Threshold=0.5')
axes[2].set_title('Predicted Probability Distribution', fontweight='bold')
axes[2].set_xlabel('P(Fake)'); axes[2].set_ylabel('Count'); axes[2].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = model.feature_importances_
importance_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6), facecolor='#F8F8F7')
ax.set_facecolor('#F8F8F7')
colors = [COLORS['fake'] if v > np.median(importances) else COLORS['bar']
          for v in importance_df['Importance']]
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
        color=colors[::-1], edgecolor='white')
ax.set_title('Feature Importances (Random Forest)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 8. Predict a New Profile
You can change these values to test any profile.

In [ ]:
def predict_profile(profile_dict):
    """Predict whether a single profile is fake or real."""
    row = pd.DataFrame([profile_dict])
    row['follower_follow_ratio'] = row['followers'] / (row['follows'] + 1)
    row['posts_per_day']         = row['posts'] / (row['account_age_days'] + 1)
    prob  = model.predict_proba(row[FEATURES])[0][1]
    label = 'FAKE 🚩' if prob >= 0.5 else 'REAL ✅'
    print(f'Prediction : {label}')
    print(f'P(fake)    : {prob:.4f}')
    return prob

# --- Example 1: Suspicious profile ---
print('=== Profile 1 (suspicious) ===')
predict_profile({
    'profile_pic': 0, 'nums_in_username': 7, 'fullname_words': 0,
    'fullname_nums': 3, 'bio_length': 5, 'external_url': 0,
    'private': 0, 'posts': 2, 'followers': 12, 'follows': 4800,
    'account_age_days': 20
})

print()

# --- Example 2: Legitimate profile ---
print('=== Profile 2 (legitimate) ===')
predict_profile({
    'profile_pic': 1, 'nums_in_username': 1, 'fullname_words': 2,
    'fullname_nums': 0, 'bio_length': 95, 'external_url': 1,
    'private': 0, 'posts': 187, 'followers': 2340, 'follows': 410,
    'account_age_days': 1200
})

## 9. To Use with Real Kaggle Data

Replace the `generate_dataset()` call with:
```python
# Download dataset from:
# https://www.kaggle.com/datasets/free4ever1/instagram-fake-spammer-genuine-accounts

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
df = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns to match this notebook if needed
df.rename(columns={'fake': 'label'}, inplace=True)
```

The rest of the notebook works unchanged.